In [2]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [37]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
groq_api_key=os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

llm=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)

from langchain_huggingface import HuggingFaceEmbeddings
embed=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [38]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents,
    embedding=embed
)

In [39]:
#async query does function execution parallel without stopping th whole program
vectorstore.similarity_search_with_score("Dog")

[(Document(id='8a8735f9-0929-4d1d-923b-44650ecbafa1', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  np.float32(1.131113)),
 (Document(id='5a1347f9-6d9f-4242-b910-a4d8817ea697', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  np.float32(1.5269936)),
 (Document(id='9b21a8c2-3dc1-42c2-b9a4-e5f2bb3787c2', metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
  np.float32(1.6571109)),
 (Document(id='42668ffd-b542-4b45-a48e-3d2f9937517e', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
  np.float32(1.6842747))]

In [40]:
### retrievers with batch() and runnablelambda

from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever=RunnableLambda(vectorstore.similarity_search).bind(k=1) #conert to retrievers
retriever.batch(["cat","dog"])


[[Document(id='5a1347f9-6d9f-4242-b910-a4d8817ea697', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='8a8735f9-0929-4d1d-923b-44650ecbafa1', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [41]:
#using asretreiver()
# as_retriever() converts a vector store into a retriever that LangChain
# chains understand.

# retriever=vectorstore.as_retriever()

# Now instead of calling:

# vectorstore.similarity_search("dog")

# you call:

# retriever.invoke("dog")

# The retriever internally performs the similarity search and returns the most relevant documents.
# It's mainly used because RAG chains (like retrieval chains) expect a Retriever, not a Vector Store.

In [42]:
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriever.batch(["cat","dog"])

[[Document(id='5a1347f9-6d9f-4242-b910-a4d8817ea697', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='8a8735f9-0929-4d1d-923b-44650ecbafa1', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [43]:
#CPT..... RPT-> is use d to identify func,pass i/p to o/p unchanged. it is used to preserved data ,mssage, dict, or inject new keys into teh pipeline wihtout chnaging the original!
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message="""
Answer this question using the provided context only.

{question}

Context:
{context}
"""

prompt=ChatPromptTemplate.from_messages([("human",message)])

rag_chain={"context":retriever,"question":RunnablePassthrough()} |prompt|llm #RPT->"Take the original input and put it into the question key."

resp=rag_chain.invoke("tell about cats")
print(resp.content)

Cats are independent pets that often enjoy their own space.
